In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_54_try_0.pickle

In [4]:
%%RecordEvent
%%time
### cell 55 ###

# Define question strings
question_of_interest_cell67 = \
    'Which of the following cloud computing platforms do you use? (Select all that apply)'
question_of_interest_old_cell67 = \
    'Which of the following cloud computing platforms do you use on a regular basis?'
question_of_interest_old_2 = \
    'Which of the following cloud computing services have you used at work or school in the last 5 years?'

# 1) Clean up the 2018 platform column names in one GPU regex replace
def _clean_2018_platform_cols(cols):
    return cols.str.replace(
        r'(Amazon Web Services \(AWS\)|Google Cloud Platform \(GCP\)|Microsoft Azure)(?! )',
        r'\1 ',
        regex=True
    )
responses_df_2018_cell10.columns = _clean_2018_platform_cols(responses_df_2018_cell10.columns)

# 2) Standardize question text across DataFrames in two GPU replace calls each
for df in (responses_df_2021, responses_df_2020, responses_df_2019_cell10, responses_df_2018_cell10):
    df.columns = (
        df.columns
          .str.replace(question_of_interest_old_cell67, question_of_interest_cell67, regex=False)
          .str.replace(question_of_interest_old_2, question_of_interest_cell67, regex=False)
    )

# 3) Pull out only the "Select all that apply" columns, strip off the prefix, drop all-null rows
def grab_subset_of_data_67(original_df, question):
    cols = [c for c in original_df.columns if question in c]
    mapper = {c: c.split('-')[-1].lstrip() for c in cols}
    return (
        original_df[cols]
          .rename(columns=mapper)
          .dropna(how='all', subset=list(mapper.values()))
    )

# 4) Combine across years with one comprehension; add a 'year' column via .assign (GPU)
def combine_subset_of_data_for_question_67(question, include_2017=False):
    dfs_map = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_df_2019_cell10,
        '2018': responses_df_2018_cell10
    }
    if include_2017:
        dfs_map['2017'] = responses_df_2017

    # Extract, rename, drop-null, and tag with year
    dfs = [
        grab_subset_of_data_67(df, question).assign(year=yr)
        for yr, df in sorted(dfs_map.items())
    ]
    combined = pd.concat(dfs, ignore_index=True)
    counts = combined.groupby('year').count().reset_index()
    return combined, counts

# 5) Run the combination
cloud_df_combined, cloud_df_combined_counts = \
    combine_subset_of_data_for_question_67(question_of_interest_cell67)

# 6) Compute percentages with one broadcasted GPU operation
#    (avoids per-row eq/sum and .copy)
year_totals = cloud_df_combined.groupby('year').size()
cloud_df_combined_percentages = (
    cloud_df_combined_counts
      .set_index('year')
      .mul(100)
      .div(year_totals, axis=0)
      .reset_index()
)

# 7) Select just the three platforms and melt, naming the variable column '' to match downstream
to_melt = ['Amazon Web Services (AWS) ', 'Google Cloud Platform (GCP) ', 'Microsoft Azure ']
cloud_select = cloud_df_combined_percentages[['year'] + to_melt]
df_cell67 = cloud_select.melt(
    id_vars=['year'],
    value_vars=to_melt,
    var_name='',    # empty string column name
    value_name='value'
)

df_cell67.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    15 non-null     object
 1           15 non-null     object
 2   value   15 non-null     float64
dtypes: float64(1), object(2)
memory usage: 658.0+ bytes
CPU times: user 706 ms, sys: 4.22 ms, total: 710 ms
Wall time: 712 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_55_try_0.pickle